<a href="https://colab.research.google.com/github/laurenkzz/1BI_InteligenciaArtificial/blob/main/Trabalho.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import pandas as pd

# Dataset (lentes de contato)
data = [
["infantil","miopia","não","reduzida","nenhuma"],
["infantil","miopia","sim","normal","gelatinosa"],
["infantil","hipermetropia","não","normal","gelatinosa"],
["infantil","hipermetropia","sim","normal","dura"],
["adolescente","miopia","não","reduzida","gelatinosa"],
["adolescente","miopia","sim","reduzida","nenhuma"],
["adolescente","miopia","não","normal","dura"],
["adolescente","hipermetropia","não","reduzida","gelatinosa"],
["adolescente","hipermetropia","sim","normal","dura"],
["adulto","miopia","não","normal","gelatinosa"],
["adulto","miopia","sim","normal","dura"],
["adulto","miopia","sim","reduzida","gelatinosa"],
["adulto","hipermetropia","não","reduzida","nenhuma"],
["adulto","hipermetropia","sim","normal","gelatinosa"],
["adulto","hipermetropia","não","normal","gelatinosa"],
]

cols = ["idade","diagnostico","astigmatismo","lacrimal","classe"]
df = pd.DataFrame(data, columns=cols)

df.head()

,idade,diagnostico,astigmatismo,lacrimal,classe
0,infantil,miopia,não,reduzida,nenhuma
1,infantil,miopia,sim,normal,gelatinosa
2,infantil,hipermetropia,não,normal,gelatinosa
3,infantil,hipermetropia,sim,normal,dura
4,adolescente,miopia,não,reduzida,gelatinosa


In [13]:
classes = df["classe"].unique()
total = len(df)

prob_classe = {}

for c in classes:
    prob_classe[c] = (len(df[df["classe"] == c]) + 1) / (total + len(classes))

prob_classe

{'nenhuma': 0.2222222222222222, 'gelatinosa': 0.5, 'dura': 0.2777777777777778}

In [14]:
def prob_condicional(df, atributo, classe):
    valores = df[atributo].unique()
    k = len(valores)

    probs = {}

    df_classe = df[df["classe"] == classe]
    total_classe = len(df_classe)

    for v in valores:
        count = len(df_classe[df_classe[atributo] == v])
        probs[v] = (count + 1) / (total_classe + k)

    return probs

atributos = ["idade","diagnostico","astigmatismo","lacrimal"]

tabelas = {}

for c in classes:
    tabelas[c] = {}
    for a in atributos:
        tabelas[c][a] = prob_condicional(df, a, c)

tabelas

{'nenhuma': {'idade': {'infantil': 0.3333333333333333,
   'adolescente': 0.3333333333333333,
   'adulto': 0.3333333333333333},
  'diagnostico': {'miopia': 0.6, 'hipermetropia': 0.4},
  'astigmatismo': {'não': 0.6, 'sim': 0.4},
  'lacrimal': {'reduzida': 0.8, 'normal': 0.2}},
 'gelatinosa': {'idade': {'infantil': 0.2727272727272727,
   'adolescente': 0.2727272727272727,
   'adulto': 0.45454545454545453},
  'diagnostico': {'miopia': 0.5, 'hipermetropia': 0.5},
  'astigmatismo': {'não': 0.6, 'sim': 0.4},
  'lacrimal': {'reduzida': 0.4, 'normal': 0.6}},
 'dura': {'idade': {'infantil': 0.2857142857142857,
   'adolescente': 0.42857142857142855,
   'adulto': 0.2857142857142857},
  'diagnostico': {'miopia': 0.5, 'hipermetropia': 0.5},
  'astigmatismo': {'não': 0.3333333333333333, 'sim': 0.6666666666666666},
  'lacrimal': {'reduzida': 0.16666666666666666, 'normal': 0.8333333333333334}}}

In [15]:
paciente = {
    "idade": "infantil",
    "diagnostico": "hipermetropia",
    "astigmatismo": "não",
    "lacrimal": "reduzida"
}

In [16]:
import numpy as np

resultados = {}

for c in classes:
    prob = prob_classe[c]

    for a in atributos:
        prob *= tabelas[c][a][paciente[a]]

    resultados[c] = prob

# Normalizar
soma = sum(resultados.values())
for c in resultados:
    resultados[c] /= soma

resultados

{'nenhuma': 0.4337306981429145,
 'gelatinosa': 0.49903674076102367,
 'dura': 0.06723256109606188}

In [17]:
def prob_lacrimal_condicional(df, classe, idade):
    subset = df[(df["classe"] == classe) & (df["idade"] == idade)]

    valores = df["lacrimal"].unique()
    k = len(valores)

    total = len(subset)

    probs = {}

    for v in valores:
        count = len(subset[subset["lacrimal"] == v])
        probs[v] = (count + 1) / (total + k)

    return probs

In [18]:
resultados_rb = {}

for c in classes:
    prob = prob_classe[c]

    prob *= tabelas[c]["idade"][paciente["idade"]]
    prob *= tabelas[c]["diagnostico"][paciente["diagnostico"]]
    prob *= tabelas[c]["astigmatismo"][paciente["astigmatismo"]]

    prob_lac = prob_lacrimal_condicional(df, c, paciente["idade"])
    prob *= prob_lac[paciente["lacrimal"]]

    resultados_rb[c] = prob

# Normalizar
soma = sum(resultados_rb.values())
for c in resultados_rb:
    resultados_rb[c] /= soma

resultados_rb

{'nenhuma': 0.44743731368127965,
 'gelatinosa': 0.386105352359911,
 'dura': 0.16645733395880938}